In [3]:
import os, re, random, pickle, json, math
import networkx as nx

In [4]:
with open("../dataset/all_materials_motif_data.pkl", 'rb') as f:
    data = pickle.load(f)

In [5]:
#Build the heterogeneous/bipartite network graph
G = nx.Graph()
for k, v in data.items():
    material = k
    #print(material)
    G.add_node(material, bipartite='material')
    motif_details = v['motif_details']
    for item in motif_details:
        for k1, v1 in item.items():
            motif = k1.removesuffix("_d").removesuffix("_p")
            G.add_node(material, bipartite='material')
            G.add_node(motif, bipartite='motif')
            csm_weight = float(v1['csm_value'])
            csm = 0.0 if csm_weight is None else round((100-float(csm_weight))/100, 3)
            if G.has_edge(material, motif):
                prev = G[material][motif].get('weight', math.inf)
                if csm < prev:
                    G[material][motif]['weight'] = csm
            else:
                G.add_edge(material, motif, weight=csm)

In [6]:
def extract_partition_nodes(G: nx.Graph, partition: str):
    nodeset = [n for n, d in G.nodes(data=True) if d["bipartite"] == partition]
    if len(nodeset) == 0:
        raise Exception(f"No nodes exist in the partition {partition}!")
    return nodeset
materials = extract_partition_nodes(G, "material")
motifs =  extract_partition_nodes(G, "motif")

In [7]:
#motif_types = extract_partition_nodes(G, "motif_type")
df = nx.to_pandas_edgelist(G, source='material', target='motif', 
                           nodelist=materials, edge_key='csm_weight')
df.to_csv("../dataset/all_materials_csm_edgelist.dat", sep = "\t", index=False)
nx.write_graphml(G, '../dataset/all_materials_network.graphml')